In [1]:
import numpy as np
import numba as nb

x = np.arange(64)

In [2]:
@nb.njit()
def pack_params_jit(Eon, Eoff, Din, Dout):

    Eon_flat = Eon.flatten()
    Eon_real = np.real(Eon_flat)
    Eon_imag = np.imag(Eon_flat)[1:]  # We can fix the phase of the first input as a reference to remove degeneracy

    Eoff_flat = Eoff.flatten()
    Eoff_real = np.real(Eoff_flat)
    Eoff_imag = np.imag(Eoff_flat)[1:]  # We can fix the phase of the first input as a reference to remove degeneracy

    # Crosstalk matrices are stored as a real vector so `least_squares`

    # Keep only non diagonal and non first-input phase terms
    # Diagonal phase terms are degenerated with E and first-input phase is used as a reference
    D_image_mask = np.array([
        [0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 1, 0, 1],
        [0, 1, 1, 0],
    ])

    Din_real = np.real(Din).flatten()
    # pack only the masked imaginary entries (non-diagonal / non-first-input)
    Din_imag = np.empty((9,))
    idx = 0
    for i in range(4):
        for j in range(4):
            if D_image_mask[i, j] == 1:
                Din_imag[idx] = np.imag(Din[i, j])
                idx += 1

    Dout_real = np.real(Dout).flatten()
    # pack only the masked imaginary entries (non-diagonal / non-first-input)
    Dout_imag = np.empty((9,))
    idx = 0
    for i in range(4):
        for j in range(4):
            if D_image_mask[i, j] == 1:
                Dout_imag[idx] = np.imag(Dout[i, j])
                idx += 1

    return np.array(list(Eon_real) + list(Eon_imag) + list(Eoff_real) + list(Eoff_imag) + list(Din_real) + list(Din_imag) + list(Dout_real) + list(Dout_imag))

@nb.njit()
def unpack_params_jit(params):

    Eon_real = params[0:4]
    Eon_imag = np.array([0] + list(params[4:7]))
    Eon = Eon_real + 1j * Eon_imag

    Eoff_real = params[7:11]
    Eoff_imag = np.array([0] + list(params[11:14]))
    Eoff = Eoff_real + 1j * Eoff_imag

    D_image_mask = np.array([
        [0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 1, 0, 1],
        [0, 1, 1, 0],
    ])

    Din_real = params[14:30].reshape(4, 4)
    Din_imag = np.zeros((4, 4)) # Initialize with zeros
    for i in range(4):
        for j in range(4):
            if D_image_mask[i, j] == 1:
                Din_imag[i, j] = params[30 + np.sum(D_image_mask[:i, :]) + np.sum(D_image_mask[i, :j])]
    Din = Din_real + 1j * Din_imag

    Dout_real = params[39:55].reshape(4, 4)
    Dout_imag = np.zeros((4, 4)) # Initialize with zeros
    for i in range(4):
        for j in range(4):
            if D_image_mask[i, j] == 1:
                Dout_imag[i, j] = params[55 + np.sum(D_image_mask[:i, :]) + np.sum(D_image_mask[i, :j])]
    Dout = Dout_real + 1j * Dout_imag

    return Eon, Eoff, Din, Dout

In [3]:
def pack_params(Eon, Eoff, Din, Dout):

    Eon_flat = Eon.flatten()
    Eon_real = np.real(Eon_flat)
    Eon_imag = np.imag(Eon_flat)[1:]  # We can fix the phase of the first input as a reference to remove degeneracy

    Eoff_flat = Eoff.flatten()
    Eoff_real = np.real(Eoff_flat)
    Eoff_imag = np.imag(Eoff_flat)[1:]  # We can fix the phase of the first input as a reference to remove degeneracy

    # Crosstalk matrices are stored as a real vector so `least_squares`

    # Keep only non diagonal and non first-input phase terms
    # Diagonal phase terms are degenerated with E and first-input phase is used as a reference
    D_image_mask = np.array([
        [0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 1, 0, 1],
        [0, 1, 1, 0],
    ])

    Din_real = np.real(Din).flatten()
    # pack only the masked imaginary entries (non-diagonal / non-first-input)
    Din_imag = np.imag(Din)[D_image_mask == 1]

    Dout_real = np.real(Dout).flatten()
    # pack only the masked imaginary entries (non-diagonal / non-first-input)
    Dout_imag = np.imag(Dout)[D_image_mask == 1]

    return np.concatenate([
        Eon_real, Eon_imag,
        Eoff_real, Eoff_imag,
        Din_real, Din_imag,
        Dout_real, Dout_imag,
    ])

def unpack_params(params):

    Eon_real = params[0:4]
    Eon_imag = np.concatenate([[0], params[4:7]])
    Eon = Eon_real + 1j * Eon_imag

    Eoff_real = params[7:11]
    Eoff_imag = np.concatenate([[0], params[11:14]])
    Eoff = Eoff_real + 1j * Eoff_imag

    D_image_mask = np.array([
        [0, 1, 1, 1],
        [0, 0, 1, 1],
        [0, 1, 0, 1],
        [0, 1, 1, 0],
    ])

    Din_real = params[14:30].reshape(4, 4)
    Din_imag = np.zeros((4, 4)) # Initialize with zeros
    Din_imag[D_image_mask == 1] = params[30:39] # Fill only the non-diagonal and non first-input phase terms
    Din = Din_real + 1j * Din_imag

    Dout_real = params[39:55].reshape(4, 4)
    Dout_imag = np.zeros((4, 4)) # Initialize with zeros
    Dout_imag[D_image_mask == 1] = params[55:64] # Fill only the non-diagonal and non first-input phase terms
    Dout = Dout_real + 1j * Dout_imag

    return Eon, Eoff, Din, Dout

In [4]:
%%timeit
Eon, Eoff, Din, Dout = unpack_params(x)
x_repacked = pack_params(Eon, Eoff, Din, Dout)

72 μs ± 22.6 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
%%timeit
Eon, Eoff, Din, Dout = unpack_params_jit(x)
x_repacked = pack_params_jit(Eon, Eoff, Din, Dout)

36.7 μs ± 8.6 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
